# 12 · Combined area against shapely

**Question.** Does Fieldwork's `summarize_combined_area` widget union polygons the way GEOS
does, and does it report the summed parts net of their holes?

This is the most genuinely independent check in the lab so far. Fieldwork unions with
**polyclip-ts** and measures with a shoelace sum; the reference unions with **GEOS** and
measures with GEOS. A union is where two libraries most plausibly differ, because it is where
coincident edges and near-degenerate intersections get decided.

The fixture is three **nested** bands' corridors — 2, 5 and 10 minutes from the Broad Street
pump. Nested is deliberate: a shorter band's corridor lies inside a longer one's, so the parts
overlap heavily and the combined area is far smaller than their sum. A fixture of disjoint
parts would have made the union trivially equal to the sum and checked nothing.

Criteria fixed before running: the combined geometry within 1 m² by symmetric difference, the
combined and summed areas each within 1 m², and the part and piece counts unchanged. One
square metre of roughly 700,000 is about one part per million.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["pyproj", "shapely", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_12_combined_area import run

report = run(Path.cwd() / "fixtures" / "combined-area-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["parts"], "parts")
print(f"combined area   fieldwork {report['fieldwork']['combinedAreaSquareMetres']:12.2f}   "
      f"shapely {report['reference']['combinedAreaSquareMetres']:12.2f}")
print(f"parts, net      fieldwork {report['fieldwork']['partsAreaSquareMetres']:12.2f}   "
      f"shapely {report['reference']['partsAreaSquareMetres']:12.2f}")
print(f"pieces          fieldwork {report['fieldwork']['combinedPieceCount']:>12}   "
      f"shapely {report['reference']['combinedPieceCount']:>12}")
print()
print(f"symmetric difference of the union {report['geometryDifferenceSquareMetres']:.3e} m2")
print(f"polyclip-ts against GEOS: two unions, independently implemented.")

## Dive deeper · the sum is not the area

The number a reader is most likely to get wrong here is not the combined area. It is the
**sum of the parts**, which looks like a total and is not one.

In [ ]:
overlap = report["overlapSquareMetres"]
parts = report["reference"]["partsAreaSquareMetres"]
combined = report["reference"]["combinedAreaSquareMetres"]
print(f"sum of the three bands' corridors   {parts:12.0f} m2")
print(f"distinct ground they cover          {combined:12.0f} m2")
print(f"counted more than once              {overlap:12.0f} m2   ({overlap / parts:.0%} of the sum)")
print()
print("These bands are nested: everything inside the 2 minute corridor is also inside the 5")
print("and the 10. Adding them counts the ground round the pump three times.")

### Where the holes come in

Fieldwork's comment on this widget records a defect found on the isochrone chain: summing
only outer rings reported **a single hole-bearing area as overlapping itself**. A 200 m square
with a 50 m hole came out as 40,000 m² of parts against 37,500 m² combined, and 2,500 m²
"covered more than once" with nothing to overlap.

That is worth reproducing, because it is the kind of error that looks like a result.

In [ ]:
import shapely
from shapely.geometry import Polygon

square = Polygon([(0, 0), (200, 0), (200, 200), (0, 200)],
                 [[(75, 75), (125, 75), (125, 125), (75, 125)]])
outer_only = Polygon(square.exterior).area
net = square.area
print(f"outer ring alone        {outer_only:9.0f} m2")
print(f"net of its hole         {net:9.0f} m2")
print(f"union of the one part   {shapely.unary_union([square]).area:9.0f} m2")
print()
print(f"Summing outer rings gives {outer_only - net:.0f} m2 'covered more than once' for a single")
print("part with nothing to overlap. Taking parts net of holes gives zero, which is the truth.")
print()
print("Reachable corridors keep their holes, so the isochrone chain hits this on every run.")

### Reading the result

`agrees` means two independent union implementations produced the same ground and the same
measurement of it. It does not mean the combined area means what a reader will take it to
mean: it is **distinct ground covered by at least one corridor**, which is not a population,
not a catchment, and not a service area.